In [19]:
import pandas as pd
import numpy as np

dataset=pd.read_csv("/synthetic_noisy_dataset_40k_15col.csv")

**Explolatory Data Analysis**

In [20]:
dataset

,age,income,experience_years,education_years,hours_worked_per_week,satisfaction_score,department,city,performance_rating,projects_completed,training_hours,remote_work_ratio,tenure_years,overtime_hours,target_salary
0,58.0,69549.87,39.0,12.0,39.3,6.14,Engineering,Sylhet,2.0,5.0,35.9,0.497,39.0,2.8,68384.37
1,52.0,60693.22,32.0,16.0,29.5,7.23,Marketing,Dhaka,5.0,5.0,6.3,0.671,32.0,2.2,65594.91
2,43.0,45906.57,25.0,19.0,58.1,7.71,HR,Rajshahi,4.0,1.0,26.9,0.187,25.0,0.5,49246.79
3,58.0,65469.20,NaN,10.0,38.9,7.45,Engineering,Chittagong,4.0,3.0,13.3,0.255,39.0,13.7,72753.03
4,45.0,60345.96,24.0,17.0,34.6,4.05,Finance,Sylhet,4.0,8.0,7.1,0.612,22.0,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40195,29.0,50688.43,9.0,19.0,NaN,7.12,Marketing,Rajshahi,4.0,7.0,70.5,0.772,8.0,NaN,38827.35
40196,61.0,76448.18,39.0,9.0,36.9,4.78,Engineering,Chittagong,2.0,6.0,14.2,0.728,39.0,0.0,64465.61
40197,54.0,59630.08,32.0,16.0,35.6,7.31,Marketing,Sylhet,4.0,4.0,NaN,0.628,31.0,6.8,59792.21
40198,26.0,50118.92,NaN,11.0,44.8,7.08,Engineering,Khulna,3.0,2.0,9.3,0.376,NaN,0.0,33740.75


In [21]:
dataset.shape

(40200, 15)

In [22]:
dataset.isnull().sum()

,0
age,1201
income,1227
experience_years,1224
education_years,1207
hours_worked_per_week,1174
satisfaction_score,1224
department,0
city,0
performance_rating,1201
projects_completed,1232


**Data Cleaning**

In [23]:
print(dataset.dtypes)
print("Duplicates:", dataset.duplicated().sum())

for col in dataset.select_dtypes(include='object').columns:
    print(f"\n{col}: {dataset[col].unique()}")

age                      float64
income                   float64
experience_years         float64
education_years          float64
hours_worked_per_week    float64
satisfaction_score       float64
department                object
city                      object
performance_rating       float64
projects_completed       float64
training_hours           float64
remote_work_ratio        float64
tenure_years             float64
overtime_hours           float64
target_salary            float64
dtype: object
Duplicates: 187

department: ['Engineering' 'Marketing' 'HR' 'Finance' 'Sales' 'hr' 'FINANCE' 'Mktg'
 'engineering' 'ENGINEERING' 'Sales ' 'sales' 'Eng' 'finance' 'SALES'
 'marketing' 'H.R.']

city: ['Sylhet' 'Dhaka' 'Rajshahi' 'Chittagong' 'chittagong' 'Khulna' 'DHAKA'
 'dhaka' 'khulna' 'rajshahi' 'Ctg' 'sylhet']


In [24]:
# ---- 1. Drop exact duplicate rows ----
print("Before dedup:", dataset.shape)
dataset = dataset.drop_duplicates()
print("After dedup:", dataset.shape)

Before dedup: (40200, 15)
After dedup: (40013, 15)


In [25]:
# ---- 2. Standardize categorical text (strip whitespace, fix casing) ----
dataset['department'] = dataset['department'].str.strip().str.lower()
dataset['city'] = dataset['city'].str.strip().str.lower()

dept_map = {
    'engineering': 'Engineering', 'eng': 'Engineering',
    'marketing': 'Marketing', 'mktg': 'Marketing',
    'hr': 'HR', 'h.r.': 'HR',
    'finance': 'Finance',
    'sales': 'Sales'
}
city_map = {
    'sylhet': 'Sylhet', 'dhaka': 'Dhaka', 'rajshahi': 'Rajshahi',
    'chittagong': 'Chittagong', 'ctg': 'Chittagong', 'khulna': 'Khulna'
}

dataset['department'] = dataset['department'].map(dept_map)
dataset['city'] = dataset['city'].map(city_map)

# sanity check - should show no NaN introduced by mapping
print(dataset['department'].isnull().sum(), dataset['city'].isnull().sum())
print(dataset['department'].unique())
print(dataset['city'].unique())


0 0
['Engineering' 'Marketing' 'HR' 'Finance' 'Sales']
['Sylhet' 'Dhaka' 'Rajshahi' 'Chittagong' 'Khulna']


/tmp/ipykernel_2224/4041592240.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset['department'] = dataset['department'].str.strip().str.lower()
/tmp/ipykernel_2224/4041592240.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset['city'] = dataset['city'].str.strip().str.lower()
/tmp/ipykernel_2224/4041592240.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the document

In [26]:
# ---- 3. Impute missing numeric values with median ----
numeric_cols = dataset.select_dtypes(include=['float64', 'int64']).columns.tolist()
for col in numeric_cols:
    dataset[col] = dataset[col].fillna(dataset[col].median())

print(dataset.isnull().sum())  # should all be 0 now

age                      0
income                   0
experience_years         0
education_years          0
hours_worked_per_week    0
satisfaction_score       0
department               0
city                     0
performance_rating       0
projects_completed       0
training_hours           0
remote_work_ratio        0
tenure_years             0
overtime_hours           0
target_salary            0
dtype: int64


/tmp/ipykernel_2224/95004861.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset[col] = dataset[col].fillna(dataset[col].median())


In [27]:
# ---- 4. Handle outliers via IQR capping (winsorizing, not deleting rows) ----
def cap_outliers_iqr(df, col, factor=1.5):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - factor * IQR
    upper = Q3 + factor * IQR
    df[col] = df[col].clip(lower, upper)
    return df

outlier_check_cols = ['age', 'income', 'experience_years', 'hours_worked_per_week',
                       'projects_completed', 'training_hours', 'tenure_years',
                       'overtime_hours', 'target_salary']

for col in outlier_check_cols:
    dataset = cap_outliers_iqr(dataset, col)

/tmp/ipykernel_2224/1214063082.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].clip(lower, upper)
/tmp/ipykernel_2224/1214063082.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].clip(lower, upper)
/tmp/ipykernel_2224/1214063082.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/use

In [28]:
# ---- 5. Sanity checks on logically invalid entries ----
# e.g. age shouldn't be negative or absurd (>100), experience_years shouldn't exceed age
print(dataset['age'].describe())
print(dataset['experience_years'].describe())
print((dataset['experience_years'] > dataset['age']).sum())  # invalid rows where exp > age

dataset.describe()

count    40013.000000
mean        43.171619
std         15.209332
min         -6.500000
25%         31.000000
50%         43.000000
75%         56.000000
max         69.000000
Name: age, dtype: float64
count    40013.000000
mean        23.529828
std         14.764484
min          0.000000
25%         11.000000
50%         23.000000
75%         36.000000
max         51.000000
Name: experience_years, dtype: float64
477


,age,income,experience_years,education_years,hours_worked_per_week,satisfaction_score,performance_rating,projects_completed,training_hours,remote_work_ratio,tenure_years,overtime_hours,target_salary
count,40013.000000,40013.000000,40013.000000,40013.000000,40013.000000,40013.000000,40013.000000,40013.000000,40013.000000,40013.000000,40013.00000,40013.000000,40013.000000
mean,43.171619,55171.878753,23.529828,13.532877,40.008171,6.979578,3.436633,4.992340,18.638268,0.499262,22.56904,5.208522,53576.513951
std,15.209332,14781.459174,14.764484,3.391872,8.010153,1.446260,1.011784,2.161573,16.152590,0.220317,14.68728,3.593788,16674.882159
min,-6.500000,12527.090000,0.000000,8.000000,19.050000,0.420000,1.000000,0.000000,0.000000,0.003000,0.00000,0.000000,2876.420000
25%,31.000000,44401.760000,11.000000,11.000000,34.800000,6.020000,3.000000,3.000000,6.000000,0.332000,10.00000,2.400000,40762.700000
50%,43.000000,54976.910000,23.000000,14.000000,40.100000,6.990000,3.000000,5.000000,13.700000,0.499000,22.00000,5.000000,53307.190000
75%,56.000000,65651.540000,36.000000,16.000000,45.300000,7.960000,4.000000,6.000000,26.900000,0.668000,35.00000,7.600000,66020.220000
max,69.000000,97526.210000,51.000000,19.000000,61.050000,10.000000,5.000000,10.500000,58.250000,0.998000,51.00000,15.400000,103906.500000


In [29]:
# Cap experience_years so it can't exceed (age - 16)
invalid_mask = dataset['experience_years'] > (dataset['age'] - 16)
print("Rows to fix:", invalid_mask.sum())

dataset.loc[invalid_mask, 'experience_years'] = dataset.loc[invalid_mask, 'age'] - 16

# Re-check
print("Remaining invalid:", (dataset['experience_years'] > dataset['age']).sum())
print(dataset['experience_years'].describe())

Rows to fix: 1181
Remaining invalid: 0
count    40013.000000
mean        23.015220
std         15.054779
min        -22.500000
25%         10.000000
50%         23.000000
75%         36.000000
max         51.000000
Name: experience_years, dtype: float64


In [30]:
print("Shape:", dataset.shape)
print("Invalid (exp > age):", (dataset['experience_years'] > dataset['age']).sum())
print("Any nulls left?", dataset.isnull().sum().sum())

Shape: (40013, 15)
Invalid (exp > age): 0
Any nulls left? 0


**Model Selection**

In [31]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Separate features and target
X = dataset.drop('target_salary', axis=1)
y = dataset['target_salary']

# Identify column types
categorical_cols = ['department', 'city']
numeric_cols = X.select_dtypes(include=['float64', 'int64']).columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

# Train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# Preprocessing: scale numeric for Linear Regression, one-hot encode categoricals for both
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

Numeric columns: ['age', 'income', 'experience_years', 'education_years', 'hours_worked_per_week', 'satisfaction_score', 'performance_rating', 'projects_completed', 'training_hours', 'remote_work_ratio', 'tenure_years', 'overtime_hours']
Categorical columns: ['department', 'city']
Train shape: (32010, 14)
Test shape: (8003, 14)


**Train Both Models**

In [32]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

In [33]:
# ---- Model 1: Linear Regression (needs scaled data) ----
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

lr_pipeline.fit(X_train, y_train)
lr_preds = lr_pipeline.predict(X_test)

In [34]:
# ---- Model 2: Random Forest (scaling not needed, but pipeline handles it fine anyway) ----
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))
])

rf_pipeline.fit(X_train, y_train)
rf_preds = rf_pipeline.predict(X_test)

print("Both models trained.")

Both models trained.


**Evaluate and Compare**

In [35]:
def evaluate(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"\n{model_name}")
    print(f"  RMSE: {rmse:.2f}")
    print(f"  MAE:  {mae:.2f}")
    print(f"  R²:   {r2:.4f}")
    return {'model': model_name, 'RMSE': rmse, 'MAE': mae, 'R2': r2}

results = []
results.append(evaluate(y_test, lr_preds, "Linear Regression"))
results.append(evaluate(y_test, rf_preds, "Random Forest"))

import pandas as pd
results_df = pd.DataFrame(results)
print("\n", results_df)


Linear Regression
  RMSE: 6289.81
  MAE:  4633.52
  R²:   0.8559

Random Forest
  RMSE: 6001.45
  MAE:  4510.10
  R²:   0.8689

                model         RMSE          MAE        R2
0  Linear Regression  6289.814602  4633.522074  0.855947
1      Random Forest  6001.454816  4510.100157  0.868852


**Model Analysis & Comparison**

After training both models, their performance was evaluated using three standard regression metrics:

**Root Mean Squared Error (RMSE):** Measures the average prediction error while giving more weight to larger errors. A lower RMSE indicates better model performance.

**Mean Absolute Error (MAE):** Measures the average absolute difference between the predicted and actual values. Lower MAE values represent more accurate predictions.

**R² Score (Coefficient of Determination):** Indicates how well the model explains the variance in the target variable. Values closer to 1 indicate better performance.

**Evaluation Results**

| Model                   |        RMSE |         MAE |   R² Score |
| :---------------------- | ----------: | ----------: | ---------: |
| Linear Regression       | **6289.81** | **4633.52** | **0.8559** |
| Random Forest Regressor | **6001.45** | **4510.10** | **0.8689** |


**Comparison**

The results show that Random Forest Regressor outperformed Linear Regression across all evaluation metrics.

**RMSE:** Random Forest achieved a lower RMSE (6001.45) compared to Linear Regression (6289.81), indicating that its predictions were closer to the actual salary values.

**MAE:** Random Forest also produced a lower MAE (4510.10), meaning it had a smaller average prediction error.

**R² Score:** The Random Forest model achieved a higher R² score (0.8689), explaining approximately 86.89% of the variance in the target variable, whereas Linear Regression explained 85.59%.

The improved performance of the Random Forest model can be attributed to its ability to capture complex nonlinear relationships and interactions among features, making it more suitable for noisy tabular datasets.


**Best Model**

Based on the evaluation metrics, Random Forest Regressor is selected as the best-performing model for this dataset. It achieved the lowest prediction errors (RMSE and MAE) and the highest R² score, demonstrating better overall predictive performance than Linear Regression.

**Conclusion**

Both models produced strong results, indicating that the selected features are effective predictors of employee salary. However, Random Forest Regressor consistently outperformed Linear Regression and is therefore recommended as the final model for this machine learning pipeline. Its robustness to noise and ability to model complex relationships make it a reliable choice for salary prediction on this dataset.